In [ ]:
#Create the GDF for the reanalysis/observations using this jupyter notebook 
##

In [3]:
sourceData='/bettik/PROJECTS/pr-regional-climate/santolam/'
#model='gpcp'#'aphro''cru'#
#model='aphro'
model='chirps'
#fileName='cru_ts4.05.1901.2020.pre.dat.nc'
#fileName='precip_mon.gpcp_v2020_1891_2019_10.nc'
#fileName='APHRO_mon_MA_025deg_V1101_EXR1.1951-2015.nc'
#fileName='cru_ts4.05.1901.2020.pre.dat.nc'
#model='esacciv20'
##fileName='snow.mon_esacci_0.5x0.5_1982-2018.nc'
#'chirps-v2.0.monthly.nc'
#'snow.mon_esacci_0.5x0.5_1982-2014_icefilled_climfilled.nc'
#var='snc_if'#'pr'
#model="HMA_SR_daily_7km_CH"
#fileName='/1999/HMA_SR_D_v01_WY1999_00_CH_lat24N32N_lon80E91E_SD_POST_MASKED_coarsen15x15.nc'

In [4]:
import numpy as np
import xarray as xr
import sys

In [5]:
#ds0= xr.open_dataset(sourceData+model+'/'+fileName)
model7='chirps'
ds0= xr.open_dataset('/bettik/PROJECTS/pr-regional-climate/santolam/'+model7+'/daily/'+'chirps-v2.0.2016.days_p05_HMA.nc')#['precip']

#ds0=xr.open_dataset('/bettik/PROJECTS/pr-regional-climate/santolam/HMA_SR_daily_7km_HK/snow_bin/'+fileName)
print(ds0)
print(ds0.coords)

<xarray.Dataset>
Dimensions:    (time: 366, longitude: 1000, latitude: 500)
Coordinates:
  * time       (time) datetime64[ns] 2016-01-01 2016-01-02 ... 2016-12-31
  * longitude  (longitude) float32 60.02 60.07 60.12 60.18 ... 109.9 109.9 110.0
  * latitude   (latitude) float32 20.02 20.07 20.12 20.17 ... 44.87 44.92 44.97
Data variables:
    precip     (time, latitude, longitude) float32 ...
Attributes: (12/17)
    CDI:               Climate Data Interface version 2.0.4 (https://mpimet.m...
    Conventions:       CF-1.6
    institution:       Climate Hazards Group.  University of California at Sa...
    title:             CHIRPS Version 2.0
    history:           Wed May 21 21:31:27 2025: cdo sellonlatbox,60,110,20,4...
    version:           Version 2.0
    ...                ...
    comments:           time variable denotes the first day of the given day....
    acknowledgements:  The Climate Hazards Group InfraRed Precipitation with ...
    ftp_url:           ftp://chg-ftpout.geog.u

In [7]:
ds0=ds0.rename({'latitude':'lat','longitude':'lon'})

In [8]:
print("It is only needed to change name of variable and eventually,the name of coordinates:\n var_obs.latitude to var_obs.lat")
print("NOTE: write x,y in minus if not it cannot read cdo")

It is only needed to change name of variable and eventually,the name of coordinates:
 var_obs.latitude to var_obs.lat
NOTE: write x,y in minus if not it cannot read cdo


In [9]:
#var_obs=ds0['scfg_interp_0.5deg_icefilled']
var='precip'
var_obs=ds0[var]
lat2d, lon2d = np.meshgrid(var_obs.lat, var_obs.lon)

lat2D = xr.DataArray(data=lat2d.transpose(),  dims=["y", "x"],
    coords=dict(x=(["x"], var_obs.lon.values), y=(["y"], var_obs.lat.values)), name='lat')

lon2D = xr.DataArray(data=lon2d.transpose(),  dims=["y", "x"],
    coords=dict(x=(["x"], var_obs.lon.values), y=(["y"], var_obs.lat.values)), name='lon')
lonlat = xr.merge([lat2D, lon2D])
lonlat.to_netcdf(sourceData+model+'/'+'grid_'+var+'_'+model+'_2d_undef.nc')

In [10]:
##

In [11]:
##Opening the undefined grid to add dummy variables and attributes 
ds=xr.open_dataset(sourceData+model+'/'+'grid_'+var+'_'+model+'_2d_undef.nc')
#ds
print('grid_'+var+'_'+model+'_2d_undef.nc')

grid_precip_chirps_2d_undef.nc


In [12]:
# Create dummy variable based on lat (or any constant field; as in make_gdf.sh C.Amory)
dummy = xr.DataArray(
    np.ones_like(ds['lat']),
    dims=("y", "x"),
    coords={"y": ds["y"], "x": ds["x"]},
    name="dummy"
)

# Assign required attributes
dummy.attrs = {
    "long_name": "dummy variable",
    "standard_name": "latitude",  
    "units": "1",
    "valid_range": (-1.e+20, 1.e+20),
    "coordinates": "lon lat"
}

# Lat and lon attributes
ds["lat"].attrs = {
    "units": "degrees_north",
    "long_name": "grid center latitude",
    "standard_name": "latitude",
    "valid_range": (-1.e+20, 1.e+20),
    "actual_range": (float(ds["lat"].min()), float(ds["lat"].max()))
}

ds["lon"].attrs = {
    "units": "degrees_east",
    "long_name": "grid center longitude",
    "standard_name": "longitude",
    "valid_range": (-1.e+20, 1.e+20),
    "actual_range": (float(ds["lon"].min()), float(ds["lon"].max()))
}

# opt: set x/y metadata
ds["x"].attrs = {
    "units": "km",
    "long_name": "X",
    "standard_name": "X"
}
ds["y"].attrs = {
    "units": "km",
    "long_name": "Y",
    "standard_name": "Y"
}

# Add the dummy variable to the dataset
ds["dummy"] = dummy

# Set global attributes
ds.attrs = {
    "title": "Grid file generated for CDO remapping - MAR model",
    "institution": "Generated via Python meshgrid",
    "history": "Created for remapping using cdo remapbil: "
}

# Save to new file
ds.to_netcdf(sourceData+model+'/'+'grid_'+var+'_'+model+'_2d_def.nc')
print(sourceData+model+'/'+'grid_'+var+'_'+model+'_2d_def.nc')

/bettik/PROJECTS/pr-regional-climate/santolam/chirps/grid_precip_chirps_2d_def.nc


In [ ]:
#ncdump -h /bettik/PROJECTS/pr-regional-climate/santolam/cru/grid_tmp_cru_2d_def.nc

In [ ]:
##
cdo -v remapbil,/bettik/PROJECTS/pr-regional-climate/santolam/cru/grid_tmp_cru_2d_def.nc -setgrid,/bettik/PROJECTS/pr-regional-climate/santolam/grid_MAR_GRc_7kmx7km_2d.nc /bettik/PROJECTS/pr-regional-climate/santolam/MARout_post/GRq/spin2/work/monthly/TTZ_mon_MARv3.14_ER5_spin2_GRq_1950-2024.nc /bettik/PROJECTS/pr-regional-climate/santolam/MARout_coarser/TTZ_mon_MARv3.14_ER5_spin2_GRq_1950-2024_cru05x05.nc
cdo -v remapbil,/bettik/PROJECTS/pr-regional-climate/santolam/cru/grid_tmp_cru_2d_def.nc -setgrid,/bettik/PROJECTS/pr-regional-climate/santolam/grid_MAR_GRi_7kmx7km_2d.nc /bettik/PROJECTS/pr-regional-climate/santolam/MARout_post/GRp/spin2/work/monthly/TTZ_mon_MARv3.14_ER5_spin2_GRp_1950-2024.nc /bettik/PROJECTS/pr-regional-climate/santolam/MARout_coarser/TTZ_mon_MARv3.14_ER5_spin2_GRp_1950-2024_cru05x05.nc

In [ ]:
cdo -v remapbil,/bettik/PROJECTS/pr-regional-climate/santolam/aphro/grid_pre_aphro_2d_def.nc -setgrid,/bettik/PROJECTS/pr-regional-climate/santolam/grid_MAR_GRc_7kmx7km_2d.nc /bettik/PROJECTS/pr-regional-climate/santolam/MARout_post/GRq/spin2/work/monthly/RF_mon_MARv3.14_ER5_spin2_GRq_1950-2024.nc /bettik/PROJECTS/pr-regional-climate/santolam/MARout_coarser/RF_mon_MARv3.14_ER5_spin2_GRq_1950-2024_aphro025x025.nc

In [ ]:
cdo -v remapbil,/bettik/PROJECTS/pr-regional-climate/santolam/aphro/grid_pre_aphro_2d_def.nc -setgrid,/bettik/PROJECTS/pr-regional-climate/santolam/grid_MAR_GRc_7kmx7km_2d.nc /bettik/PROJECTS/pr-regional-climate/santolam/MARout_post/GRq/spin2/work/monthly/SF_mon_MARv3.14_ER5_spin2_GRq_1950-2024.nc /bettik/PROJECTS/pr-regional-climate/santolam/MARout_coarser/SF_mon_MARv3.14_ER5_spin2_GRq_1950-2024_aphro025x025.nc

In [ ]:
cdo -v remapbil,/bettik/PROJECTS/pr-regional-climate/santolam/aphro/grid_pre_aphro_2d_def.nc -setgrid,/bettik/PROJECTS/pr-regional-climate/santolam/grid_MAR_GRi_7kmx7km_2d.nc /bettik/PROJECTS/pr-regional-climate/santolam/MARout_post/GRp/spin2/work/monthly/SF_mon_MARv3.14_ER5_spin2_GRp_1950-2024.nc /bettik/PROJECTS/pr-regional-climate/santolam/MARout_coarser/SF_mon_MARv3.14_ER5_spin2_GRp_1950-2024_aphro025x025.nc

In [ ]:
cdo -v remapbil,/bettik/PROJECTS/pr-regional-climate/santolam/aphro/grid_pre_aphro_2d_def.nc -setgrid,/bettik/PROJECTS/pr-regional-climate/santolam/grid_MAR_GRi_7kmx7km_2d.nc /bettik/PROJECTS/pr-regional-climate/santolam/MARout_post/GRp/spin2/work/monthly/RF_mon_MARv3.14_ER5_spin2_GRp_1950-2024.nc /bettik/PROJECTS/pr-regional-climate/santolam/MARout_coarser/RF_mon_MARv3.14_ER5_spin2_GRp_1950-2024_aphro025x025.nc